# 01 — Vehicle Dynamics from the Contact Patch

> **Engineering question:** How can the car accelerate, brake, or turn if the road only interacts with it through four tire contact patches?

This notebook is the computational companion to the whiteboard lesson. The governing mechanics come first; Python is used to reproduce the hand calculation, check physical invariants, and connect the result to reviewed WUFR reference data.

**Workflow:** physical question → system boundary → FBD → derivation → explicit calculation → sanity check → engineering interpretation.

## 1. Load the reviewed WUFR reference state

The notebook does **not** contain copied vehicle constants. It loads the same reviewed source records used by the suspension software through the education-facing `VehicleReference` facade.

The reference is design-intent/prototype authority for this lesson, not installed/as-built authority.

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

from IPython.display import Markdown, display
import matplotlib.pyplot as plt


def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not find repository root from the current working directory")


# Allow the notebook to run directly from a repository checkout even when the
# active kernel/environment has not installed the package with `pip install -e .`.
ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pssd_vehicle import load_vehicle_reference


SELECTOR = ROOT / "configurations/education/WUFR27_EDUCATION_BASELINE_V0.toml"
vehicle = load_vehicle_reference(SELECTOR)

print(f"Repository:        {ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Vehicle selector:  {vehicle.selector_id}")
print(f"Reference state:   {vehicle.state_id}")
print(f"Whole-vehicle:     {vehicle.whole_vehicle_adapter_id}")
print(f"Gravity record:    {vehicle.gravity_record_id}")


In [ ]:
rows = [
    ("Total mass, m", vehicle.total_mass_kg, "kg"),
    ("Gravity, g", vehicle.g_mps2, "m/s²"),
    ("Wheelbase, L", vehicle.geometry.wheelbase_m, "m"),
    ("CG → front axle, l_f", vehicle.geometry.cg_to_front_axle_m, "m"),
    ("CG → rear axle, l_r", vehicle.geometry.cg_to_rear_axle_m, "m"),
]

table = [
    "| Quantity | Value | Unit |",
    "|---|---:|---|",
    *[f"| {name} | {value:.6f} | {unit} |" for name, value, unit in rows],
]
display(Markdown("\n".join(table)))


## 2. Coordinate convention and system boundary

Use the repository vehicle frame throughout:

- **+x** forward
- **+y** vehicle left
- **+z** upward

For the complete-car free-body diagram, the motor, drivetrain, springs, dampers, suspension links, and steering system are **inside** the system boundary. Their forces are internal to this model. Gravity and the tire/road forces cross the boundary and are external.

Changing the system boundary changes which forces are external. Later, when we isolate an upright or control arm, suspension-member forces will become external unknowns.

## 3. Side-view free-body diagram

For the static axle-load problem, collapse the four vertical tire reactions into one front-axle reaction and one rear-axle reaction. The side-view model contains three external vertical forces:

$$F_{z,F}, \qquad F_{z,R}, \qquad m\,g$$

with

$$L = l_f + l_r.$$

In [ ]:
L = vehicle.geometry.wheelbase_m
l_f = vehicle.geometry.cg_to_front_axle_m
l_r = vehicle.geometry.cg_to_rear_axle_m

x_rear = 0.0
x_cg = l_r
x_front = L

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot([x_rear, x_front], [0.45, 0.45], linewidth=3)
ax.scatter([x_rear, x_cg, x_front], [0.45, 0.78, 0.45], zorder=3)

ax.annotate("", xy=(x_rear, 1.35), xytext=(x_rear, 0.05), arrowprops={"arrowstyle": "->", "linewidth": 2})
ax.annotate(r"$F_{z,R}$", xy=(x_rear, 1.38), ha="center")

ax.annotate("", xy=(x_front, 1.35), xytext=(x_front, 0.05), arrowprops={"arrowstyle": "->", "linewidth": 2})
ax.annotate(r"$F_{z,F}$", xy=(x_front, 1.38), ha="center")

ax.annotate("", xy=(x_cg, -0.15), xytext=(x_cg, 1.15), arrowprops={"arrowstyle": "->", "linewidth": 2})
ax.annotate(r"$m g$", xy=(x_cg, -0.18), ha="center", va="top")
ax.annotate("CG", xy=(x_cg, 0.82), ha="center", va="bottom")

ax.annotate("", xy=(x_cg, -0.48), xytext=(x_rear, -0.48), arrowprops={"arrowstyle": "<->"})
ax.annotate(r"$l_r$", xy=((x_rear + x_cg) / 2, -0.55), ha="center", va="top")
ax.annotate("", xy=(x_front, -0.48), xytext=(x_cg, -0.48), arrowprops={"arrowstyle": "<->"})
ax.annotate(r"$l_f$", xy=((x_cg + x_front) / 2, -0.55), ha="center", va="top")

ax.text(x_rear, 0.28, "rear contact line", ha="center", va="top")
ax.text(x_front, 0.28, "front contact line", ha="center", va="top")
ax.set_xlim(-0.18 * L, 1.18 * L)
ax.set_ylim(-0.85, 1.65)
ax.set_axis_off()
ax.set_title("Static two-axle side-view free-body diagram")
plt.show()


## 4. Derive the static axle loads before calculating them

Assume the car is motionless on level ground, with no aerodynamic force or other external support. Then linear and angular acceleration are zero.

### Vertical force equilibrium

$$\sum F_z = 0$$

$$F_{z,F} + F_{z,R} - m\,g = 0$$

so

$$F_{z,F} + F_{z,R} = m\,g. \tag{1}$$

Equation (1) gives the **total** vertical support, but it is one equation with two unknown axle reactions. We need a moment equation to determine how the load is distributed.

### Moment equilibrium about the rear contact line

Choose the rear contact line as point $R$. For any force,

$$\mathbf{M}_R = \mathbf{r}\times\mathbf{F}.$$

The line of action of $F_{z,R}$ passes through $R$, so its position vector from $R$ is zero:

$$\mathbf{r}_{R\rightarrow rear}=\mathbf{0}$$

and therefore

$$\mathbf{M}_R(F_{z,R})=\mathbf{0}\times\mathbf{F}_{z,R}=\mathbf{0}.$$

**The rear reaction is not zero; its moment arm about the chosen point is zero.**

The front reaction has perpendicular moment arm $L$, while the vertical weight force has perpendicular moment arm $l_r$. Choosing one pitch-moment direction as positive gives

$$\sum M_R = m\,g\,l_r - F_{z,F}L = 0.$$

Therefore,

$$\boxed{F_{z,F}=m\,g\frac{l_r}{L}}$$

and from Equation (1),

$$\boxed{F_{z,R}=m\,g\frac{l_f}{L}}.$$

Notice that CG height does not appear in this **static vertical-load** balance. Weight acts vertically, so its perpendicular distance from the rear contact line is the horizontal distance $l_r$.

## 5. WUFR hand calculation in Python

The following cell is intentionally explicit. There is no `static_axle_loads()` solver hiding the mechanics. Each line maps directly to the equations above.

In [ ]:
m_kg = vehicle.total_mass_kg
g_mps2 = vehicle.g_mps2
L_m = vehicle.geometry.wheelbase_m
l_f_m = vehicle.geometry.cg_to_front_axle_m
l_r_m = vehicle.geometry.cg_to_rear_axle_m

weight_N = m_kg * g_mps2
front_load_N = weight_N * l_r_m / L_m
rear_load_N = weight_N * l_f_m / L_m

front_fraction = front_load_N / weight_N
rear_fraction = rear_load_N / weight_N

results = [
    ("Total weight", weight_N, "N"),
    ("Front axle load", front_load_N, "N"),
    ("Rear axle load", rear_load_N, "N"),
    ("Front fraction", 100.0 * front_fraction, "%"),
    ("Rear fraction", 100.0 * rear_fraction, "%"),
]

table = [
    "| Result | Value | Unit |",
    "|---|---:|---|",
    *[f"| {name} | {value:.3f} | {unit} |" for name, value, unit in results],
]
display(Markdown("\n".join(table)))


## 6. Sanity checks and comparison with the reviewed scale state

A model result is not useful merely because Python returned a number. Check invariants that must be true from the physics:

- $l_f+l_r=L$
- $F_{z,F}>0$ and $F_{z,R}>0$ for a CG between the axles
- $F_{z,F}+F_{z,R}=m\,g$
- front fraction + rear fraction = 1

Then compare the predicted axle fractions with the reviewed four-corner scale state. This is a **consistency check, not independent validation**, because the reviewed planar CG location was derived from that scale state.

In [ ]:
assert math.isclose(l_f_m + l_r_m, L_m, rel_tol=0.0, abs_tol=1e-12)
assert front_load_N > 0.0
assert rear_load_N > 0.0
assert math.isclose(front_load_N + rear_load_N, weight_N, rel_tol=0.0, abs_tol=1e-9)
assert math.isclose(front_fraction + rear_fraction, 1.0, rel_tol=0.0, abs_tol=1e-12)

scale = vehicle.scale_state
scale_front_fraction = scale.front_fraction
scale_rear_fraction = scale.rear_fraction

assert math.isclose(front_fraction, scale_front_fraction, rel_tol=0.0, abs_tol=1e-12)
assert math.isclose(rear_fraction, scale_rear_fraction, rel_tol=0.0, abs_tol=1e-12)

comparison = [
    ("Front", front_fraction, scale_front_fraction),
    ("Rear", rear_fraction, scale_rear_fraction),
]

table = [
    "| Axle | Equilibrium model | Reviewed scales |",
    "|---|---:|---:|",
    *[f"| {axle} | {100*model:.3f}% | {100*measured:.3f}% |" for axle, model, measured in comparison],
]
display(Markdown("\n".join(table)))

corner_names = [name.replace("_", " ").title() for name in scale.corner_order]
corner_table = [
    "| Scale corner | Reading |",
    "|---|---:|",
    *[f"| {name} | {load:.1f} lb |" for name, load in zip(corner_names, scale.corner_load_lb)],
    f"| **Total** | **{scale.total_load_lb:.1f} lb** |",
]
display(Markdown("\n".join(corner_table)))

print("Physics checks passed.")


## 7. Limiting case — predict before running

Move the CG to the midpoint between the axles while keeping total mass and wheelbase fixed.

**Before running the next cell:** what should the front/rear load split become, and why?

In [ ]:
l_f_center_m = L_m / 2.0
l_r_center_m = L_m / 2.0

front_center_N = weight_N * l_r_center_m / L_m
rear_center_N = weight_N * l_f_center_m / L_m

print(f"Centered CG front load: {front_center_N:.3f} N")
print(f"Centered CG rear load:  {rear_center_N:.3f} N")

assert math.isclose(front_center_N, weight_N / 2.0, rel_tol=0.0, abs_tol=1e-12)
assert math.isclose(rear_center_N, weight_N / 2.0, rel_tol=0.0, abs_tol=1e-12)


## 8. The first cornering question

Return to Newton's second law in the lateral direction:

$$\sum F_y = m\,a_y.$$

If the lateral acceleration magnitude is $n\,g$, then

$$\sum F_y = n\,m\,g.$$

This tells us the **total lateral road-force demand**. It does not yet tell us how the four tires share that force, what slip angle is required, or whether the tires can produce it.

In [ ]:
lateral_cases_g = (1.0, 1.5)
rows = []
for lateral_g in lateral_cases_g:
    a_y_mps2 = lateral_g * g_mps2
    total_lateral_force_N = m_kg * a_y_mps2
    rows.append((lateral_g, a_y_mps2, total_lateral_force_N))

table = [
    "| Lateral acceleration | a_y | Required total lateral force |",
    "|---:|---:|---:|",
    *[
        f"| {lateral_g:.1f} g | {a_y:.3f} m/s² | {force:.3f} N |"
        for lateral_g, a_y, force in rows
    ],
]
display(Markdown("\n".join(table)))


## Engineering takeaway

The useful result from this lesson is not a memorized axle-weight formula. It is the mechanics workflow:

> **choose the system boundary → draw the FBD → write force/moment balance → solve → check limiting cases → compare with data → state what the model omitted**

For this simple static model, axle load is proportional to the **opposite-side CG lever arm**. The model determines front/rear totals but intentionally throws away left/right corner information.

### Check yourself

1. Why does $F_{z,R}$ disappear when moments are taken about the rear contact line?
2. If the CG moves rearward while $m$ and $L$ remain fixed, which axle gains static load? Explain from the moment balance.
3. Why can this two-axle side-view model not predict the four individual scale readings?

### Handoff to Lesson 2

At 1.5 g we now know how much total lateral force the car requires. The next question is the important one:

> **How does a tire actually generate lateral force?**